# 05. Подготовка датасета

Этот ноутбук показывает, как из исходного файла `data/raw/movies.csv` получается очищенный датасет `data/processed/output.csv`. Ноутбук ничего не перезаписывает: все операции выполняются в памяти.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_PATH = PROJECT_ROOT / "data" / "movies.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "output.csv"

raw_df = pd.read_csv(RAW_PATH)
clean_df = pd.read_csv(CLEAN_PATH)

print("Исходный датасет:", raw_df.shape)
print("Очищенный датасет:", clean_df.shape)
raw_df.head()

## Пропуски в исходном датасете

Главная проблема исходного файла — неполные значения `budget` и `gross`. Для обучения модели кассовой выручки эти поля критичны: `gross` является target, а `budget` — один из ключевых признаков.

In [ ]:
missing_raw = pd.DataFrame({
    "missing": raw_df.isna().sum(),
    "missing_pct": (raw_df.isna().mean() * 100).round(2),
}).sort_values("missing", ascending=False)
missing_clean = pd.DataFrame({
    "missing": clean_df.isna().sum(),
    "missing_pct": (clean_df.isna().mean() * 100).round(2),
}).sort_values("missing", ascending=False)

display(missing_raw)
display(missing_clean)

## Воспроизводимая логика очистки

Ниже показана безопасная логика очистки без сохранения результата:

1. привести `gross` и `budget` к числовому типу;
2. удалить строки без `gross`;
3. удалить строки без `budget`;
4. удалить строки, где `gross <= 0` или `budget <= 0`;
5. удалить оставшиеся строки с пропусками в обязательных колонках.

In [ ]:
required_columns = [
    "name", "rating", "genre", "year", "released", "score", "votes",
    "director", "writer", "star", "country", "budget", "gross", "company", "runtime",
]

prepared_df = raw_df.copy()
prepared_df["gross"] = pd.to_numeric(prepared_df["gross"], errors="coerce")
prepared_df["budget"] = pd.to_numeric(prepared_df["budget"], errors="coerce")

steps = []
steps.append({"step": "raw", "rows": len(prepared_df)})

prepared_df = prepared_df.dropna(subset=["gross"])
steps.append({"step": "drop missing gross", "rows": len(prepared_df)})

prepared_df = prepared_df.dropna(subset=["budget"])
steps.append({"step": "drop missing budget", "rows": len(prepared_df)})

prepared_df = prepared_df[(prepared_df["gross"] > 0) & (prepared_df["budget"] > 0)].copy()
steps.append({"step": "keep positive gross and budget", "rows": len(prepared_df)})

prepared_df = prepared_df.dropna(subset=required_columns)
steps.append({"step": "drop missing required columns", "rows": len(prepared_df)})

pd.DataFrame(steps)

## Сравнение с текущим `output.csv`

Если количество строк совпадает, значит текущий `output.csv` соответствует базовой логике очистки. Даже если порядок строк или типы немного отличаются, важна воспроизводимость правил отбора данных.

In [ ]:
comparison = pd.DataFrame([
    {"dataset": "prepared in memory", "rows": len(prepared_df), "columns": prepared_df.shape[1]},
    {"dataset": "output.csv", "rows": len(clean_df), "columns": clean_df.shape[1]},
])
comparison

## Проверка минимального бюджета

В веб-сервисе добавлена нижняя граница бюджета `300 000 USD`, потому что значения ниже этого уровня находятся на краю или вне устойчивого диапазона обучающей выборки. Это снижает риск неадекватной экстраполяции модели.

In [ ]:
budget = clean_df["budget"]
summary = budget.describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50]).to_frame("budget")
display(summary)

print("Фильмов с бюджетом ниже 300 000 USD:", int((budget < 300_000).sum()))
print("Доля таких фильмов, %:", round((budget < 300_000).mean() * 100, 2))

## Вывод

`output.csv` является очищенным датасетом без пропусков в обязательных полях. Он подходит для обучения модели прогнозирования `gross`, но содержит выбросы по очень малым бюджетам и сильное смещение в сторону зарубежного рынка.